# Performing a genome-wide association study (GWAS) and meta-analysis:
from data preparation to analysis of results

# Objectives
In this workshop you will learn the basic skills needed to perform genetic association studies, from file format manipulation to filtering, single-point association and visualisation of results. We will be working with a simulated  phenotype and a QCed dataset.


We will be using <code>plink</code> to run the association and R to standardise our phenotype and visualise the results.
You can find a manual and command reference of <code>plink</code> [here](https://www.cog-genomics.org/plink/1.9/) and [here](https://www.cog-genomics.org/plink/2.0/) depending on the version.

In R we will be using the package <code>data.table</code>, which provides a lot of useful commands, such as <code>fread</code> for fast reading in of large files. If you want to find out more about <code>data.table</code> and its perks, you can do so [here](https://cran.r-project.org/web/packages/data.table/vignettes/datatable-intro.html) and [here](https://cran.r-project.org/web/packages/data.table/data.table.pdf).


### Files, Software, Libraries

### Google Drive

Let us set up the connection with Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# this is the file path to your google drive (/content/drive/My Drive/) followed up by the data path

dir_path='/content/drive/My Drive/Complex_Traits/3_Workshop_Genetic_Association_MetaAnalysis/'


### Sofware and libraries

Install the rpy2 package that allows us to run R from a python notebook

In [ ]:
!pip install numpy
!pip install pandas
!pip install rpy2>=3.8.0

In [ ]:
import os # python related package to list files in the defined directory
import rpy2.ipython
%load_ext rpy2.ipython

In [ ]:
%%R
dir_path = '/content/drive/My Drive/Complex_Traits/3_Workshop_Genetic_Association_MetaAnalysis/data'
setwd(paste0(dir_path))

### Downloading PLINK software (both versions 1.9 and 2.0)

In [ ]:
!wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20210606.zip && unzip plink_linux_x86_64_20210606.zip && rm prettify toy.* LICENSE
!rm *.zip

### Downloading PLINK 2.0 software

In [ ]:
# Download PLINK 2.0
!wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip -O plink2.zip && unzip plink2.zip && rm plink2.zip


### Test PLINK 2.0 installation

In [ ]:
%%bash
# Make plink2 executable
chmod +x plink2

# Verify PLINK 2.0 installation
./plink2 --version

# Step 1: Phenotypes


In large-scale sample collections you will often find that some individuals do not have a data entry for your phenotype of interest.

This is normally not a problem, as long as it doesn't affect too many samples. Association programs like <code>plink</code> and <code>SNPTEST</code> treat certain values (usually NA or -9) as missing phenotypes and exclude the corresponding samples automatically.

Nevertheless, it is good practice to check how many samples have missing phenotypes before running an association, and also make sure that their phenotype is set to an accepted missing value.  

We will work with the file <code>Data_1kG_SimuPheno.txt</code> which contains sample IDs, population information and phenotype values for a quantitative and binary phenotype. We will start by focusing on the quantitative phenotype (<code>Y_Quanti</code>).

<br>

<b>Question 1:</b>  In the <code>data</code> folder you will find the file called <code>Data_1kG_SimuPheno.txt</code>. Using the command line, count the number of "NA" phenotypes.

<br>




In [ ]:
# use this tab and try to solve the task by youself

In [ ]:
#@title Solution
%%bash

awk '$4=="NA"{print}' Data_1kG_SimuPheno.txt | wc -l

# alternatively, you can use grep
# grep 'NA$' Data_1kG_SimuPheno.txt | wc -l

## Data Cleaning and Standardisation



When working with quantitative traits, such as height, BMI or blood lipids, you will often find that the measurements in your cohort do not follow a normal distribution.
____

This could simply be due to the way your samples were collected - e.g. height follows an approximately normal distrbution at the population level; however, if you randomly pick 100 people, you might end up with a skewed distribution, simply because by chance there is a disproportionate number of very tall people in your sample.
____

On the other hand, some traits naturally follow a non-normal distribution, such as certain blood metabolites. Since linear regression models and numerous statistical tests rely on the assumption of a normally distributed response variable (the phenotype), it is important to inspect your phenotype measurements and, if necessary, standardise them before conducting a GWAS.

<b>Question 2:</b>  Load the <code>Data_1kG_SimuPheno.txt</code> file in R and look at the distribution of the phenotype values using a boxplot. Are there any outliers?

In [ ]:
# use this tab and try to solve the task by youself





In [ ]:
#@title Solution
%%R
library(data.table)
dataPheno=fread("Data_1kG_SimuPheno.txt")
boxplot(dataPheno$Y_Quanti, xlab="Phenotype", main="Raw phenotype")

From the plot, we cannot see any clear outlier. Depending on the type of trait you are looking at, you will have different criteria for outlier definition. For phsyiological or anthropometric measurements it is a good rule of thumb to look for negative values or ones that are biologically impossible (for example, it's highly unlikely to find someone who is 900m tall!).  



<b>Question 3:</b>  To look at the distribution of the data, use a histogram to represent the quantitative phenotype. What do you observe?

In [ ]:
# use this tab and try to solve the task by youself





In [ ]:
#@title Solution
%%R
hist(dataPheno$Y_Quanti, xlab="Phenotype", main="Raw phenotype")

There seems to be a bimodal distribution of the phenotype with two groups of individuals.

<b>Question 4:</b> Look at the distribution of phenotypes values within each of the large population groups. Do you observe any trend?

In [ ]:
# use this tab and try to solve the task by youself



In [ ]:
#@title Solution
%%R
#Compare the phenotype values according the large populations
boxplot(dataPheno$Y_Quanti ~ dataPheno$LargePop)

We see a clear difference of phenotype values between North Europe and South Europe. We will investigate what is the impact of such distribution on GWAS.

<b>Question 5:</b> Is the phenotype normally distributed within each of the large population? Try to plot the distribution in each population using the functions <code>subset</code> and <code>par(mfrow()).</code>



In [ ]:
# use this tab and try to solve the task by youself



In [ ]:
#@title Solution
%%R
#Distribution of the phenotypes in each of the large population
par(mfrow=c(1,2))
hist(subset(dataPheno, LargePop=="NEUR")$Y_Quanti, xlab="Phenotype", main="North EUR")
hist(subset(dataPheno, LargePop=="SEUR")$Y_Quanti, xlab="Phenotype", main="South EUR")

To evaluate the impact of the bimodal distribution on association tests, we will perform a GWAS on the whole set of individuals. As the distribution of the phenotype in the whole group is bimodal, we will first normalize the phenotype.   
Usually, normalizing this kind of phenotype is not the best idea, we will evaluate later on the impact of this bimodal distribution.

<b>Question 6:</b> Use the inverse-normal transformation to standardise the phenotype.


In [ ]:
# use this tab and try to solve the task by youself




In [ ]:
#@title Solution
%%R

# we use an inverse-normal transformation to standardise our phenotype
dataPheno$stdnorm=qnorm((rank(dataPheno$Y_Quanti,na.last="keep"))/(nrow(dataPheno)))

# let's take a look at the transformed phenotype
hist(dataPheno$stdnorm,xlab="Phenotype", main="Transformed phenotype")

<b>Bonus Question:</b> What does the <code>na.last</code> parameter in the <code>rank</code> function above do? Why is it important to set it to <code>"keep"</code>?



Now that we have transformed our phenotype, we just need to save it to a <code>Plink</code> [compatible format](https://www.cog-genomics.org/plink/1.9/input#pheno). The <code>Plink</code> pheno file is similar to a .fam file, with the first two columns being individual and family IDs; the third column should contain the phenotype values. NA values are coded as -9 in <code>Plink</code> phenotype files.

<b>Question 7:</b> Modify the NA values to -9 and save the phenotype file with the three required columns to <code>Data_1kG_pheno_final.txt</code>.



In [ ]:
# use this tab and try to solve the task by youself






In [ ]:
#@title Solution
%%R
#Replace NA values with -9 for Plink
PhenoFile <- dataPheno[,.(0,IID,stdnorm)]
PhenoFile[is.na(stdnorm), stdnorm := -9]
# save file
fwrite(PhenoFile,
            "Data_1kG_pheno_final.txt",
            col.names=F,row.names=F,
            quote=F,
            sep="\t")

We will also perform a GWAS within each subpopution.   
<b>Question 8:</b> Save the corresponding phenotype files in each of the large population under  <code>Data_1kG_pheno_final_NEUR.txt</code> and <code>Data_1kG_pheno_final_SEUR.txt</code>

In [ ]:
# use this tab and try to solve the task by youself



In [ ]:
#@title Solution
%%R
#Replace NA values with -9 for Plink
PhenoFile_NEUR <- subset(dataPheno, LargePop == "NEUR")[,.(0,IID,Y_Quanti)]
PhenoFile_NEUR[is.na(Y_Quanti), Y_Quanti := -9]
# save file
fwrite(PhenoFile_NEUR,
            "Data_1kG_pheno_final_NEUR.txt",
            col.names=F,row.names=F,
            quote=F,
            sep="\t")

####SEUR
#Replace NA values with -9 for Plink
PhenoFile_SEUR <- subset(dataPheno, LargePop == "SEUR")[,.(0,IID,Y_Quanti)]
PhenoFile_SEUR[is.na(Y_Quanti), Y_Quanti := -9]
# save file
fwrite(PhenoFile_SEUR,
            "Data_1kG_pheno_final_SEUR.txt",
            col.names=F,row.names=F,
            quote=F,
            sep="\t")

# Step 2: GWAS on the full dataset

To run the association analysis, we will use data based on the 1000 Genomes project in the EUR population that have already been QCed (/content/drive/My Drive/Complex_Traits/data_GWAS/1kg_phase1_all_EUR_QCed), and the phenotype file we created above.

<br>
<div class="alert alert-warning"><b>Location:</b> Input files for this step are located in the <code>data</code> folder.</div>
__

<b>Question 9:</b> We want to perform genetic association in our cohort with our transformed phenotype stored in the <code>Data_1kG_pheno_final.txt</code> file. Run the association using <code>Plink</code>. We will also perform the association tests within each of the large population <code>NEUR</code> and <code>SEUR</code>. For each of the GWAS, find the option in <code>Plink</code> to compute the lambda value (or genomic inflation factor of each GWAS).   
__

<div class="alert alert-warning"><b>Note:</b> These commands might take a few minutes to run.</div>

In [ ]:
# try yourself
!./plink2 # add the necessary arguments and name the output as QuantiPheno_1kG_assoc
#Name the files QuantiPheno_1kG_NEUR_assoc and QuantiPheno_1kG_SEUR_assoc for the analyses in each population


In [ ]:

#!./plink2 --bfile QCed_Data --assoc --pheno phenofile.txt --out QuantiPheno_1kG_assoc

In [ ]:
#@title Solution on the whole dataset
!./plink2 --bfile "/content/drive/My Drive/Complex_Traits/data_GWAS/1kg_phase1_all_EUR_QCed" --glm allow-no-covars --adjust gc --pheno Data_1kG_pheno_final.txt --out QuantiPheno_1kG_assoc

In [ ]:
#@title Solution within large populations
!./plink2 --bfile "/content/drive/My Drive/Complex_Traits/data_GWAS/1kg_phase1_all_EUR_QCed" --glm allow-no-covars --adjust gc --pheno Data_1kG_pheno_final_NEUR.txt --out QuantiPheno_1kG_NEUR_assoc
!./plink2 --bfile "/content/drive/My Drive/Complex_Traits/data_GWAS/1kg_phase1_all_EUR_QCed" --glm allow-no-covars --adjust gc --pheno Data_1kG_pheno_final_SEUR.txt --out QuantiPheno_1kG_SEUR_assoc

<b>Question 10:</b> By looking at the plink output or the log file, can you find the lambda values of the three GWAS? What do you observe?


The plink command produces an output file in the [`qassoc`](https://www.cog-genomics.org/plink2/formats#qassoc) format ("q" stands for quantitative, as our trait is not binary).



**N.B.:** For Plink, "A1" is usually the minor allele and also set as the risk allele.


> PLINK association reports are very readable for the human eye, but not so for other programs (mainly because Plink adds multiple spaces to display rows in an orderly fashion, rather than tabs). Let's take some time to make our file more computer-friendly.

> We want to remove multiple whitespace characters and convert the file to a tab-delimited format. We will do it on the GWAS output ran on the full cohort to visually investigate the inflation in R.


In [ ]:
%%bash
head QuantiPheno_1kG_assoc.PHENO1.glm.linear

# Step 3: Visualisation of results

<br>
<div class="alert alert-warning"><b>Location:</b> Input files for this step are located in the <code>data</code> folder.</div>


Now that you have run the association, you will want to see whether there are any significant associations in your data.
There are two main plots generated after an association run:

* **The Quantile-Quantile (QQ)** plot is essentially a diagnostic plot. It compares the distribution of p-values against a uniform (expected) distribution. Any deviation from the expected is indicative of an issue (sample relatedness, population stratification, non-normality of phenotype values, etc...). If the associaiton p-values are systematically lower (i.e. more significant) than expected, we refer to this as "inflation".
* **The Manhattan Plot** displays the $-log_{10}$ of the SNP p-values across the genome and allows to easily spot signals (peaks).


<br>

<b>Question 11:</b>  In R, use the <code>fread</code> function from the <code>data.table</code> package to read the file. Plot a QQ-plot for the association p-values using  <code>qq</code> from the <code>qqman</code> package on the full 1kG dataset. Do you expect to see an inflation given the previous analyses performed?  
__  
<div class="alert alert-warning"><b>Note:</b> You will need to install the packages. As the output files are quite big, we will do the visual inspection only for the GWAS on the full dataset.</div>



In [ ]:
# use this tab and try yourself!



In [ ]:
%%R
install.packages("qqman")
library(data.table)
library(qqman)

In [ ]:
# use this tab and try yourself!



In [ ]:
#@title QQplot Solution
%%R
GWAS_ALL=fread("QuantiPheno_1kG_assoc.PHENO1.glm.linear")
qq(GWAS_ALL$P, main="QQ plot - Whole 1kG")


We can clearly see the inflation on the QQ-plot. Let's explore on the Manhattan plot if there is a specific region of the genome significant in the GWAS.  

<b>Question 12:</b>  In R, use the function <code>manhattan</code> from the <code>qqman</code> package on the full 1kG dataset.  
__  
<div class="alert alert-warning"><b>Note:</b> Be careful about the column names.</div>

In [ ]:
# use this tab and try yourself!



In [ ]:
#@title Manhattan plot Solution
%%R
manhattan(GWAS_ALL,main="Manhattan plot - Whole 1kG", chr = "#CHROM", bp = "POS", snp = "ID")

## What are the peaks?

In the Manhattan plot, we can see that there are multiple locus reaching genome-wide signifcance ($p<5*10e^{-8}$). Given the inflation we observed in this GWAS, it is likely that some of those signals are false positives.   
We can see one clear peak on chromosome 2. To get a closer look at the results, you can use the following command which sorts the results by p-value and displays the first few lines of the sorted file.




In [ ]:
%%R
head(GWAS_ALL[order(GWAS_ALL$P, decreasing = FALSE),])

<b>Question 13:</b> In the function <code>order</code>, what is the argument <code>decreasing</code> doing?  
__  
   
<b>Question 14:</b> What is the rsID of the top SNP? What is its direction of effect and its chromosome position?   
This region contains the *Lactase* gene. This gene is known to be highly stratified in Europe. This support the hypothesis of a genomic inflation related to population stratification.


# Step 4: GWAS Meta-analysis


In the first GWAS we performed, we observed an inflation, both in the value of the genomic inflation factor and on the QQ-plot. This inflation is probably due to population stratification given the signal around the *Lactase* gene and the observation that phenotype values were very different between the two subpopulations in our inflation.

<b>Question 15:</b> What are the lambda values obtained when performing the GWAS in NEUR and SEUR separately? Do you observe any inflation?




When performing the association within each of the large population, there is no genomic inflation. However, we would still want to analyse all the individuals to maximise statistical power and to detect true association signals linked to the phenotype. One way to do this is through meta-analysis which combines multiple datasets.

We will now meta-analyse the GWAS from the two large populations using <code>METAL</code>.   
__  
<div class="alert alert-warning"><b>Note:</b> We have used plink2.0 to perform the GWAS as the output files can be directly used in METAL, unlike plink1.9 outputs which require additional formatting. For example, the information on effect alleles and associated frequences are not available in the output files.</div>


To run the meta-analysis, we will use the following parameters file:

### Configuration de `metal.par` pour les sorties PLINK2 (`.glm.linear`)

In [ ]:
%%bash

# Create the MetaAnalysis directory if it doesn't exist
mkdir -p MetaAnalysis

# --- Updated parameters for METAL, compatible with PLINK2 --glm output ---
echo "
AVERAGEFREQ ON
MINMAXFREQ ON
ADDFILTER A1_FREQ >= 0.01
ADDFILTER A1_FREQ <= 0.99
MARKER ID
ALLELE OMITTED A1
FREQ A1_FREQ
EFFECT BETA
SCHEME STDERR
STDERR SE
PVALUE P
# Explicitly map OBS_CT from PLINK2 output to METAL's N column
COLUMN N as OBS_CT
USESTRAND OFF
MAXWARNINGS 100000
# GENOMICCONTROL ON # Uncomment if you want METAL to apply genomic control
" > MetaAnalysis/metal_plink2.par

# Add PROCESS commands for each PLINK2 output file
echo "PROCESS QuantiPheno_1kG_NEUR_assoc.PHENO1.glm.linear" >> MetaAnalysis/metal_plink2.par
echo "PROCESS QuantiPheno_1kG_SEUR_assoc.PHENO1.glm.linear" >> MetaAnalysis/metal_plink2.par

echo "
OUTFILE MetaAnalysis/QuantiPheno_plink2 .TBL
ANALYZE HETEROGENEITY
" >> MetaAnalysis/metal_plink2.par

# Display the generated metal_plink2.par file content
cat MetaAnalysis/metal_plink2.par

In [ ]:
%%bash
#Installation of the software
wget http://csg.sph.umich.edu/abecasis/metal/download/Linux-metal.tar.gz
tar -zxvf Linux-metal.tar.gz

In [ ]:
%%bash
## check that it installed okay
./generic-metal/metal
## when not specifying a command metal will print the help message which shows the available commands

We can now use <code>METAL</code> to run the meta-analysis on the parameters file which has been created.

In [ ]:
#@title Run `METAL`
#This step can take a few minutes to run
%%bash
./generic-metal/metal MetaAnalysis/metal_plink2.par

## Investigation of meta-analysis results
To investigate the meta-analysis results, we will first format a little bit the output files, and especially retrieve the rsIDs of the variants from the previosu <code>Plink</code> files.

In [ ]:
%%R
# Load the METAL meta-analysis results
GWAS_META = fread("MetaAnalysis/QuantiPheno_plink21.TBL", sep="\t")# Rename columns to match qqman's expectations if necessary
# Convert 'P-value' to numeric and assign to 'P' column (expected by qqman) and remove NAs
GWAS_META$P <- as.numeric(GWAS_META[['P-value']])
GWAS_META <- GWAS_META[!is.na(GWAS_META$P), ]
#Convert alleles to upper cases
GWAS_META$Allele1 <- toupper(GWAS_META$Allele1)
GWAS_META$Allele2 <- toupper(GWAS_META$Allele2)

In [ ]:
%%R
# METAL output columns are usually: MarkerName, Allele1, Allele2, Freq1, Weight, Zscore, Pvalue, Direction, HetISq, HetChiSq, HetDf, HetPvalue
# We need a column for Chromosome (CHR), Position (BP), and SNP identifier (SNP)
# Let's merge with rs number from plink output file
Plink.out <- fread("QuantiPheno_1kG_NEUR_assoc.PHENO1.glm.linear")
# Correct syntax for renaming column in data.table for use of manhattan plot
setnames(Plink.out, "#CHROM", "CHR")
setnames(Plink.out, "POS", "BP")

In [ ]:
%%R
# Merge with rs number to get chr and pos from rs
GWAS_META <- merge(GWAS_META, Plink.out[,c('ID', 'CHR', 'BP', 'A1', 'OMITTED')], by.x = c("MarkerName", "Allele1", "Allele2"), by.y = c("ID", "A1", "OMITTED"))
GWAS_META$SNP <- GWAS_META$MarkerName

In [ ]:
%%R
#Save the Meta-analysis file with rs numbers
fwrite(GWAS_META, "MetaAnalysis/QuantiPheno_plink2_withrs.TBL", sep="\t")

<b>Question 16:</b> Draw the QQ-plot and the manhattan-plot of the meta-analysis results. What do you observe?

In [ ]:
# use this tab and try yourself!



In [ ]:
#@title Solution
%%R
# QQ Plot for Meta-analysis
qq(GWAS_META$P, main="QQ plot - Meta-analysis")

In [ ]:
#@title Solution
%%R
# Manhattan Plot for Meta-analysis
manhattan(GWAS_META, main="Manhattan plot - Meta-analysis")

## Investigation of meta-analysis results
To investigate the benefits of meta-analysis over GWAS per population, we will compare the results of both approaches.


<b>Question 17:</b> How many variants present genome-wide significant association from the meta-analysis and is the direction consistent in the two individual GWAS?


In [ ]:
# use this tab and try yourself!



In [ ]:
#@title Solution
%%R
subset(GWAS_META, P<5e-8)

<b>Question 18:</b> Retrieve the rsIDs of these variants, were they significant in the two individual GWAS?

In [ ]:
# use this tab and try yourself!



In [ ]:
#@title Solution
%%R
#Import GWAS per population
GWAS_NEUR <- fread("QuantiPheno_1kG_NEUR_assoc.PHENO1.glm.linear")
GWAS_SEUR <- fread("QuantiPheno_1kG_SEUR_assoc.PHENO1.glm.linear")

In [ ]:
#@title Solution
%%R
sigMeta.IDs <- subset(GWAS_META, P<5e-8)$MarkerName
print(sum(subset(GWAS_NEUR, ID %in% sigMeta.IDs)$P<5e-8))
print(sum(subset(GWAS_SEUR, ID %in% sigMeta.IDs)$P<5e-8))

As for the GWAS results, it is possible to further investigate the significant variants from the meta-analysis. This will be done in another lecture.